# Day 3 Session 2 -- Deployment and Scaling Demos

These demos cover the operational skills for taking a consulting AI application from prototype to production. You are the engineer; the consultants who query the system are your users.

**Demos in this notebook:**
1. AI Service -- API design with request validation, error handling, structured responses
2. Semantic Caching -- reduce costs on recurring consulting queries
3. Monitoring and Logging -- structured metrics for review and SLA tracking
4. Model Routing -- route queries to cheaper models when complexity is low
5. Cost Tracking -- token usage and budget alerts per model

In [1]:
!pip install -q openai chromadb python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


---
## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import json
import time
import hashlib
import logging
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np

print("All imports successful!")

All imports successful!


---
## Demo 1: Designing a Consulting AI API Service

Production consulting AI applications need a clean API layer. This demo builds a service class that wraps the LLM with request validation, error handling, and structured responses -- the same patterns you would use in a FastAPI endpoint for Knowledge Search, Engagement Assistant, or Client Briefing Generator APIs.

In [3]:
# Demo 1 - McKinsey Consulting AI API Service Design

@dataclass
class ConsultingAIRequest:
    """Validated request for the McKinsey Consulting AI service."""
    prompt: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 500
    temperature: float = 0.7
    request_id: str = ""
    engagement_id: str = "ENG-DEFAULT"
    service_type: str = "knowledge_search"
    
    def __post_init__(self):
        if not self.request_id:
            self.request_id = hashlib.md5(f"{self.prompt}{time.time()}".encode()).hexdigest()[:8]
        if self.max_tokens < 1 or self.max_tokens > 4096:
            raise ValueError("max_tokens must be between 1 and 4096")
        if self.temperature < 0 or self.temperature > 2:
            raise ValueError("temperature must be between 0 and 2")
        valid_services = ["knowledge_search", "engagement_assistant", "briefing_generator"]
        if self.service_type not in valid_services:
            raise ValueError(f"service_type must be one of {valid_services}")

@dataclass
class ConsultingAIResponse:
    """Structured response from the McKinsey Consulting AI service."""
    content: str
    model: str
    request_id: str
    tokens_used: int
    latency_ms: float
    engagement_id: str = ""
    service_type: str = ""
    status: str = "success"
    error: Optional[str] = None

class McKinseyAIService:
    """Production consulting AI service with validation and error handling.
    
    Supports three API endpoints:
    - Knowledge Search: Query the internal knowledge base
    - Engagement Assistant: Help with active engagements
    - Briefing Generator: Generate client-ready briefing documents
    """
    
    SYSTEM_PROMPTS = {
        "knowledge_search": "You are McKinsey's Knowledge Search AI. Provide concise, framework-driven answers drawing on consulting best practices, industry analyses, and strategic frameworks (Porter's Five Forces, 7S, MECE, etc.).",
        "engagement_assistant": "You are McKinsey's Engagement Assistant AI. Help consultants with active client engagements by providing structured analysis, workstream planning, and deliverable outlines.",
        "briefing_generator": "You are McKinsey's Client Briefing Generator. Create executive-ready briefing content that is structured, data-driven, and actionable for C-suite audiences."
    }
    
    def __init__(self):
        self.client = openai.OpenAI()
    
    def invoke(self, request: ConsultingAIRequest) -> ConsultingAIResponse:
        start = time.time()
        system_prompt = self.SYSTEM_PROMPTS.get(request.service_type, self.SYSTEM_PROMPTS["knowledge_search"])
        try:
            response = self.client.chat.completions.create(
                model=request.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": request.prompt}
                ],
                max_tokens=request.max_tokens,
                temperature=request.temperature
            )
            latency = (time.time() - start) * 1000
            return ConsultingAIResponse(
                content=response.choices[0].message.content,
                model=request.model,
                request_id=request.request_id,
                tokens_used=response.usage.total_tokens,
                latency_ms=round(latency, 2),
                engagement_id=request.engagement_id,
                service_type=request.service_type
            )
        except Exception as e:
            latency = (time.time() - start) * 1000
            return ConsultingAIResponse(
                content="", model=request.model,
                request_id=request.request_id,
                tokens_used=0, latency_ms=round(latency, 2),
                engagement_id=request.engagement_id,
                service_type=request.service_type,
                status="error", error=str(e)
            )

# Test the McKinsey Consulting AI Service
service = McKinseyAIService()

req = ConsultingAIRequest(
    prompt="What are the key elements of a McKinsey market entry strategy for a healthcare client?",
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150")), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")),
    engagement_id="ENG-2024-0892",
    service_type="knowledge_search"
)
resp = service.invoke(req)

print(f"Request ID:    {resp.request_id}")
print(f"Engagement:    {resp.engagement_id}")
print(f"Service Type:  {resp.service_type}")
print(f"Status:        {resp.status}")
print(f"Model:         {resp.model}")
print(f"Tokens:        {resp.tokens_used}")
print(f"Latency:       {resp.latency_ms}ms")
print(f"Response:      {resp.content}")

Request ID:    efa0c630
Engagement:    ENG-2024-0892
Service Type:  knowledge_search
Status:        success
Model:         gpt-4o-mini
Tokens:        372
Latency:       5321.83ms
Response:      A McKinsey market entry strategy for a healthcare client typically involves the following key elements, structured around a framework-driven approach:

1. **Market Assessment**:
   - **Market Size and Growth**: Analyze the current market size, growth rates, and future projections.
   - **Market Segmentation**: Identify and segment the market based on demographics, needs, and behaviors.
   - **Regulatory Environment**: Assess healthcare regulations, compliance requirements, and potential barriers to entry.

2. **Competitive Analysis (Porter's Five Forces)**:
   - **Threat of New Entrants**: Evaluate barriers to entry, such as capital requirements and regulatory hurdles.
   - **Bargaining Power of Suppliers**: Analyze the power of suppliers in terms of pricing and availability of healthcare produc

---
## Demo 2: Semantic Caching for Consulting Queries

LLM calls are expensive and slow. Consultants across engagements frequently ask similar market research questions. Semantic caching stores previous responses and returns them for similar (not just identical) queries -- reducing costs by 30-70% when multiple teams research overlapping industries or frameworks.

In [4]:
# Demo 2 - Semantic Caching for McKinsey Consulting Queries

class SemanticCache:
    """Cache that matches by semantic similarity, not exact string match.
    
    Designed for consulting queries where different users
    often ask similar market research and industry analysis questions.
    """
    
    def __init__(self, similarity_threshold=0.92):
        self.client = openai.OpenAI()
        self.cache = []  # List of (embedding, query, response)
        self.threshold = similarity_threshold
        self.hits = 0
        self.misses = 0
    
    def _embed(self, text):
        return self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[text]
        ).data[0].embedding
    
    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def get(self, query):
        """Check cache for a semantically similar consulting query."""
        query_emb = self._embed(query)
        best_sim, best_response = 0, None
        
        for cached_emb, cached_query, cached_response in self.cache:
            sim = self._cosine_sim(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_response = (cached_query, cached_response)
        
        if best_sim >= self.threshold:
            self.hits += 1
            return {"hit": True, "similarity": best_sim,
                    "cached_query": best_response[0], "response": best_response[1]}
        self.misses += 1
        return {"hit": False, "similarity": best_sim}
    
    def put(self, query, response):
        """Store a consulting query-response pair in cache."""
        emb = self._embed(query)
        self.cache.append((emb, query, response))

# Test with McKinsey consulting queries
cache = SemanticCache(similarity_threshold=0.90)
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# First query -- a market research question (cache miss)
q1 = "What are the key drivers of digital transformation in the healthcare industry?"
result = cache.get(q1)
print(f"Query: {q1}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")

# Call LLM and cache the response
answer = llm.invoke([HumanMessage(content=q1)]).content
cache.put(q1, answer)
print(f"Cached response: {answer[:100]}...")

# Similar consulting query -- should hit cache (different phrasing, same research question)
q2 = "What is driving digital transformation in healthcare?"
result = cache.get(q2)
print(f"\nQuery: {q2}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
if result['hit']:
    print(f"Matched to: {result['cached_query']}")

# Different industry analysis -- should miss (unrelated topic)
q3 = "What are the main cost reduction levers in automotive manufacturing?"
result = cache.get(q3)
print(f"\nQuery: {q3}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
print(f"\nStats: {cache.hits} hits, {cache.misses} misses")

Query: What are the key drivers of digital transformation in the healthcare industry?
Cache: False (sim=0.0000)
Cached response: Digital transformation in the healthcare industry is driven by several key factors, including:

1. *...

Query: What is driving digital transformation in healthcare?
Cache: True (sim=0.9223)
Matched to: What are the key drivers of digital transformation in the healthcare industry?

Query: What are the main cost reduction levers in automotive manufacturing?
Cache: False (sim=0.3078)

Stats: 1 hits, 2 misses


---
## Demo 3: Monitoring and Structured Logging

In production, engineering and management teams need visibility into AI performance. Structured logging captures every request, response quality metrics, token count, latency, and errors -- enabling SLA tracking, partner review of AI-generated analyses, and performance monitoring across engagements.

In [5]:
# Demo 3 - Monitoring and Structured Logging for Consulting AI

class ConsultingAIMonitor:
    """Monitors consulting AI usage with structured logging and metrics.
    
    Tracks response quality for review, analysis accuracy,
    framework coverage, and per-engagement performance.
    """
    
    def __init__(self):
        self.logs = []
        self.metrics = defaultdict(list)
    
    def log_request(self, request_id, model, prompt, response, tokens, latency_ms,
                    engagement_id="", service_type="", status="success"):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "request_id": request_id,
            "engagement_id": engagement_id,
            "service_type": service_type,
            "model": model,
            "prompt_preview": prompt[:100],
            "response_preview": response[:100] if response else "",
            "tokens": tokens,
            "latency_ms": latency_ms,
            "status": status
        }
        self.logs.append(entry)
        self.metrics["latency"].append(latency_ms)
        self.metrics["tokens"].append(tokens)
        self.metrics["models"].append(model)
        self.metrics["service_types"].append(service_type)
        return entry
    
    def get_summary(self):
        if not self.logs:
            return "No requests logged."
        latencies = self.metrics["latency"]
        tokens = self.metrics["tokens"]
        models = self.metrics["models"]
        service_types = self.metrics["service_types"]
        return {
            "total_requests": len(self.logs),
            "avg_latency_ms": round(np.mean(latencies), 2),
            "p95_latency_ms": round(np.percentile(latencies, 95), 2),
            "total_tokens": sum(tokens),
            "avg_tokens_per_query": round(np.mean(tokens), 2),
            "model_distribution": {m: models.count(m) for m in set(models)},
            "service_type_distribution": {s: service_types.count(s) for s in set(service_types) if s},
            "error_rate": sum(1 for l in self.logs if l["status"] != "success") / len(self.logs),
            "partner_review_ready": sum(1 for l in self.logs if l["status"] == "success")
        }

# Test with consulting queries across different service types
monitor = ConsultingAIMonitor()
client = openai.OpenAI()

consulting_queries = [
    ("What is McKinsey's 7S framework?", "knowledge_search", "ENG-2024-0101"),
    ("Summarize the competitive landscape in European fintech.", "engagement_assistant", "ENG-2024-0205"),
    ("What are the top 3 cost reduction levers for a telecom client?", "knowledge_search", "ENG-2024-0312"),
    ("Draft an executive summary for the digital transformation engagement.", "briefing_generator", "ENG-2024-0205"),
    ("Compare market entry strategies: greenfield vs. acquisition vs. JV.", "knowledge_search", "ENG-2024-0101"),
]

for prompt, svc_type, eng_id in consulting_queries:
    start = time.time()
    resp = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=[{"role": "user", "content": prompt}], max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80")))
    latency = (time.time() - start) * 1000
    req_id = hashlib.md5(prompt.encode()).hexdigest()[:8]
    
    entry = monitor.log_request(
        request_id=req_id, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        prompt=prompt, response=resp.choices[0].message.content,
        tokens=resp.usage.total_tokens, latency_ms=round(latency, 2),
        engagement_id=eng_id, service_type=svc_type
    )
    print(f"[{entry['request_id']}] [{entry['service_type']:22s}] {entry['prompt_preview'][:45]:45s} | {entry['tokens']}tok | {entry['latency_ms']}ms")

print("\n--- Consulting AI Performance Summary ---")
summary = monitor.get_summary()
for k, v in summary.items():
    print(f"  {k}: {v}")

[a748d646] [knowledge_search      ] What is McKinsey's 7S framework?              | 318tok | 6666.61ms
[a6875927] [engagement_assistant  ] Summarize the competitive landscape in Europe | 317tok | 4826.43ms
[894a8b92] [knowledge_search      ] What are the top 3 cost reduction levers for  | 322tok | 7046.27ms
[d2c3528b] [briefing_generator    ] Draft an executive summary for the digital tr | 317tok | 5530.74ms
[c6c487e0] [knowledge_search      ] Compare market entry strategies: greenfield v | 321tok | 5583.81ms

--- Consulting AI Performance Summary ---
  total_requests: 5
  avg_latency_ms: 5930.77
  p95_latency_ms: 6970.34
  total_tokens: 1595
  avg_tokens_per_query: 319.0
  model_distribution: {'gpt-4o-mini': 5}
  service_type_distribution: {'engagement_assistant': 1, 'knowledge_search': 3, 'briefing_generator': 1}
  error_rate: 0.0
  partner_review_ready: 5


---
## Demo 4: Model Routing by Consulting Complexity

Not every consulting query needs GPT-4o. A model router classifies query complexity and routes accordingly: simple fact lookups (framework definitions, industry stats) go to gpt-4o-mini, while complex strategy analyses (market entry evaluation, M&A synergy modeling) go to gpt-4o -- cutting costs by 40-60% with minimal quality loss on routine queries.

In [6]:
# Demo 4 - Model Routing by Consulting Complexity

class ConsultingModelRouter:
    """Routes consulting queries to appropriate models based on complexity.
    
    Routing logic:
    - simple: Framework definitions, industry stats, quick fact lookups -> gpt-4o-mini
    - medium: Industry comparisons, trend summaries, workstream planning -> gpt-4o-mini
    - complex: Multi-dimensional strategy analysis, M&A evaluation -> gpt-4o
    """
    
    MODEL_TIERS = {
        "simple": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "medium": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "complex": {"model": "gpt-4o", "cost_per_1k": 0.005}
    }
    
    def __init__(self):
        self.classifier = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
    
    def classify_complexity(self, query):
        """Classify consulting query complexity to determine model tier."""
        response = self.classifier.invoke([
            SystemMessage(content="""Classify this consulting query's complexity. Return ONLY one word:
- simple: factual lookups, framework definitions, industry statistics, quick answers
- medium: industry comparisons, trend summaries, moderate analysis, workstream planning
- complex: multi-step strategy analysis, M&A evaluation, market entry planning, competitive response design"""),
            HumanMessage(content=query)
        ])
        complexity = response.content.strip().lower()
        if complexity not in self.MODEL_TIERS:
            complexity = "medium"
        return complexity
    
    def route(self, query):
        """Route consulting query to the appropriate model."""
        complexity = self.classify_complexity(query)
        tier = self.MODEL_TIERS[complexity]
        return {"complexity": complexity, "model": tier["model"], "cost_per_1k": tier["cost_per_1k"]}

# Test with consulting queries of varying complexity
router = ConsultingModelRouter()
consulting_queries = [
    "What does MECE stand for?",
    "Compare the competitive landscape of European vs. US fintech markets across regulation, adoption, and profitability.",
    "Design a post-merger integration plan for a $2B healthcare acquisition including synergy capture, org design, and Day 1 readiness.",
    "What is Porter's Five Forces framework?",
    "Develop a market entry strategy for a Fortune 500 retailer entering Southeast Asian e-commerce with build, buy, and partner options."
]

for q in consulting_queries:
    result = router.route(q)
    print(f"[{result['complexity']:7s}] -> {result['model']:12s} (${result['cost_per_1k']}/1k) | {q[:70]}")

[simple ] -> gpt-4o-mini  ($0.00015/1k) | What does MECE stand for?
[complex] -> gpt-4o       ($0.005/1k) | Compare the competitive landscape of European vs. US fintech markets a
[complex] -> gpt-4o       ($0.005/1k) | Design a post-merger integration plan for a $2B healthcare acquisition
[simple ] -> gpt-4o-mini  ($0.00015/1k) | What is Porter's Five Forces framework?
[complex] -> gpt-4o       ($0.005/1k) | Develop a market entry strategy for a Fortune 500 retailer entering So


---
## Demo 5: Token Usage and Cost Tracking

Tracking costs is critical for budget management. This demo builds a cost tracker that monitors token usage per model, calculates running costs, and provides spending alerts when budget thresholds are approached.

In [7]:
# Demo 5 - Token Usage and Cost Tracking

class CostTracker:
    """Tracks token usage and costs across models."""
    
    PRICING = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "text-embedding-3-small": {"input": 0.00002, "output": 0}
    }
    
    def __init__(self, budget_limit=1.0):
        self.budget_limit = budget_limit
        self.records = []
    
    def record(self, model, input_tokens, output_tokens, request_id=""):
        pricing = self.PRICING.get(model, {"input": 0.001, "output": 0.002})
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1000
        entry = {
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost": cost,
            "request_id": request_id
        }
        self.records.append(entry)
        total = sum(r["cost"] for r in self.records)
        if total > self.budget_limit * 0.8:
            print(f"  WARNING: Budget at {total/self.budget_limit*100:.1f}% (${total:.6f}/${self.budget_limit})")
        return entry
    
    def get_report(self):
        total_cost = sum(r["cost"] for r in self.records)
        by_model = defaultdict(lambda: {"requests": 0, "input_tokens": 0, "output_tokens": 0, "cost": 0})
        for r in self.records:
            by_model[r["model"]]["requests"] += 1
            by_model[r["model"]]["input_tokens"] += r["input_tokens"]
            by_model[r["model"]]["output_tokens"] += r["output_tokens"]
            by_model[r["model"]]["cost"] += r["cost"]
        return {"total_cost": total_cost, "total_requests": len(self.records),
                "budget_remaining": self.budget_limit - total_cost, "by_model": dict(by_model)}

# Test
tracker = CostTracker(budget_limit=0.05)
client = openai.OpenAI()

queries = [
    ("gpt-4o-mini", "What is RAG?"),
    ("gpt-4o-mini", "List three vector databases."),
    ("gpt-4o", "Design a production RAG architecture with caching and monitoring."),
    ("gpt-4o-mini", "What is cosine similarity?"),
]

for model, prompt in queries:
    resp = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}], max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
    )
    entry = tracker.record(model, resp.usage.prompt_tokens, resp.usage.completion_tokens)
    print(f"[{model:12s}] {prompt[:40]:40s} | ${entry['cost']:.6f}")

print("\n--- Cost Report ---")
report = tracker.get_report()
print(f"Total cost: ${report['total_cost']:.6f}")
print(f"Budget remaining: ${report['budget_remaining']:.6f}")
for model, stats in report['by_model'].items():
    print(f"  {model}: {stats['requests']} reqs, {stats['input_tokens']+stats['output_tokens']} tokens, ${stats['cost']:.6f}")

[gpt-4o-mini ] What is RAG?                             | $0.000145
[gpt-4o-mini ] List three vector databases.             | $0.000092
[gpt-4o      ] Design a production RAG architecture wit | $0.004590
[gpt-4o-mini ] What is cosine similarity?               | $0.000182

--- Cost Report ---
Total cost: $0.005008
Budget remaining: $0.044992
  gpt-4o-mini: 3 reqs, 724 tokens, $0.000418
  gpt-4o: 1 reqs, 318 tokens, $0.004590
